In [11]:
# Iport Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)

In [3]:
# Upload the Data
df=pd.read_csv('fmnist_small.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0


In [14]:
# Extract Features values and Target values
X=df.iloc[:, 1:].to_numpy()
y=df.iloc[:, 0].to_numpy()

In [15]:
# Split the Dataset into train and testset
X_train,X_test, y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [17]:
# Scaling the features
X_train=X_train/255.0
X_test=X_test/255.0

In [19]:
# Create CustomDataset Class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features=torch.tensor(features, dtype=torch.float32)
        self.labels=torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [20]:
# Create train dataset object
train_dataset=CustomDataset(X_train, y_train)

In [21]:
# Create test Dataset object
test_dataset=CustomDataset(X_test, y_test)

In [22]:
# Create train and test dataloader
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)

In [26]:
# Define Neural Network Class

class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        return self.model(x)

In [30]:
# Set Learning Rate and epochs
epochs=100
learning_rate=0.1

In [28]:
# Initiate the model
model=MyNN(X_train.shape[1])

# Loss Function
criterion=nn.CrossEntropyLoss()

# Optimizer
optimizer=optim.SGD(model.parameters(), lr=learning_rate)

In [31]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.3371541194121044
Epoch: 2 , Loss: 0.7805597092707952
Epoch: 3 , Loss: 0.6585829015572866
Epoch: 4 , Loss: 0.5937397322058677
Epoch: 5 , Loss: 0.5473931573828061
Epoch: 6 , Loss: 0.5099762096007665
Epoch: 7 , Loss: 0.49445449461539587
Epoch: 8 , Loss: 0.4564973184466362
Epoch: 9 , Loss: 0.42291766117016477
Epoch: 10 , Loss: 0.4065241925418377
Epoch: 11 , Loss: 0.3883439432581266
Epoch: 12 , Loss: 0.3795236760874589
Epoch: 13 , Loss: 0.35835386052727697
Epoch: 14 , Loss: 0.34707953115304313
Epoch: 15 , Loss: 0.33660522758960726
Epoch: 16 , Loss: 0.31245248546202975
Epoch: 17 , Loss: 0.30227764884630837
Epoch: 18 , Loss: 0.30452503795425095
Epoch: 19 , Loss: 0.278316719258825
Epoch: 20 , Loss: 0.30384748260180156
Epoch: 21 , Loss: 0.2696054550508658
Epoch: 22 , Loss: 0.26033384064833326
Epoch: 23 , Loss: 0.2559136170645555
Epoch: 24 , Loss: 0.24903214300672213
Epoch: 25 , Loss: 0.23929615510006746
Epoch: 26 , Loss: 0.23226683504879475
Epoch: 27 , Loss: 0.229033245419462

In [32]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.8391666666666666
